### Alternative Data Preparation for Wind Offshore ML-Model: interpolation

In this notebook, I prepare an alternative version of the offshore wind dataset with a **15-minute weather grid obtained by interpolation**.

Unlike the original notebook, I do not multiply each hourly row four times and do not add a `minute_step` feature. Instead, the missing quarter-hour weather states are estimated between consecutive hourly forecasts. The final table keeps the standard set of **14 point-wise `power_proxy` features** and adds the binary `is_interpolated` flag:

- `0` — the timestamp comes directly from the original hourly forecast;
- `1` — the weather row was reconstructed for a 15-, 30-, or 45-minute timestamp.

Two final variants are created: the standard dataset without temporal deltas and an optional dataset with T−1/T+1 hourly deltas.

In [ ]:
# Imports
import os

import numpy as np
import pandas as pd
import pyrootutils
from dotenv import load_dotenv

# Needed to load variables from the .env file into the environment
load_dotenv()

# Determining the project root based on the .gitignore file
PROJECT_ROOT = pyrootutils.setup_root(
    search_from=__file__ if "__file__" in locals() else ".",
    indicator=".gitignore",
    pythonpath=True,
)

# Determining data directories
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw_data"
CLEAN_DATA_DIR = PROJECT_ROOT / "data" / "clean_data"
DATASETS_DIR = PROJECT_ROOT / "data" / "datasets"

# Setting display options for pandas DataFrames
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

# Set an integer here only for a quick development run; None processes the full history
RUN_LIMIT = None


### Loading the three prepared data sources

This alternative notebook starts from the cleaned hourly weather table produced in `1_data_preparation_wind_off.ipynb`. The original notebook remains unchanged.

The weather table is deduplicated using the forecast run, valid timestamp, and measurement point as a composite key. Interpolation is always performed **inside a single forecast run and a single point**, so information is never mixed between forecast vintages or geographic locations.

In [ ]:
# Importing cleaned hourly weather forecasts, actual generation, and installed capacity
clean_offshore_df = pd.read_parquet(
    CLEAN_DATA_DIR / 'offshore_singleruns_only_2024-03-14_to_2026-06-21.parquet'
)
actual_generation_data = pd.read_parquet(
    RAW_DATA_DIR / 'gen_act_per_type_2024-03-13_to_2026-06-11.parquet'
)
wind_offshore_capacity = pd.read_parquet(
    RAW_DATA_DIR / 'wind_offshore_installed_capacity_2024-01-01_to_2026-07-01.parquet'
)

# Removing exact key duplicates that can appear at boundaries of downloaded chunks
weather_key = ['dt_run_utc', 'dt_utc', 'point_id']
duplicate_count = clean_offshore_df.duplicated(weather_key).sum()

clean_offshore_df = (
    clean_offshore_df
    .sort_values(weather_key)
    .drop_duplicates(weather_key, keep='last')
    .reset_index(drop=True)
)

# Optional restriction for a quick development run
if RUN_LIMIT is not None:
    selected_runs = clean_offshore_df['dt_run_utc'].drop_duplicates().iloc[:RUN_LIMIT]
    clean_offshore_df = clean_offshore_df[
        clean_offshore_df['dt_run_utc'].isin(selected_runs)
    ].copy()

print(f"Removed weather duplicates: {duplicate_count}")
print(f"Weather rows after deduplication: {len(clean_offshore_df)}")
print(f"Measurement points: {clean_offshore_df['point_id'].nunique()}")
print(f"Forecast runs: {clean_offshore_df['dt_run_utc'].nunique()}")


In [ ]:
# Preview of data from all three sources
print('Weather data:\n',
      clean_offshore_df[['dt_run_utc', 'dt_utc', 'point_id', 'w_speed', 'w_dir', 'temp']].sample(5),
      '\n\n============================================\n\n')

print('Installed capacity wind offshore:\n',
      wind_offshore_capacity.head(),
      '\n\n============================================\n\n')

print('Actual offshore generation:\n',
      actual_generation_data[['gen_wind_off_act']].head())


### Choice of interpolation strategy

There is no observed 15-minute weather series in the source data, so no interpolation method can be validated directly at the target resolution. I therefore use two safeguards:

1. **Physical transformation before interpolation.** Wind speed and direction are converted to Cartesian wind-vector components. Interpolating angles directly would produce a false turn through 180° between, for example, 359° and 1°.
2. **Masked-hour validation.** Some original hourly observations are temporarily hidden and reconstructed from their neighbours. Linear time interpolation and shape-preserving cubic interpolation (PCHIP) can then be compared on observations that were not used for reconstruction.

For the current data, masked-hour validation shows a small but consistent advantage for shape-preserving cubic interpolation (PCHIP), so it is used as the production default. Linear time interpolation remains available as a conservative alternative that cannot overshoot between two hourly values.

The recommended pipeline interpolates the **meteorological variables first** and calculates `power_proxy = density × w_speed³` afterwards. A direct proxy interpolation is included in the validation only as a benchmark: due to the cubic transformation, it is less physically interpretable.

In [ ]:
# Interpolation configuration
INTERPOLATION_METHOD = 'pchip'  # Supported values: 'time' and 'pchip'
INTERPOLATION_FREQUENCY = '15min'
VALIDATION_RUNS = 40

if INTERPOLATION_METHOD not in {'time', 'pchip'}:
    raise ValueError("INTERPOLATION_METHOD must be either 'time' or 'pchip'")

# Continuous variables interpolated together with Cartesian wind components
CONTINUOUS_WEATHER_COLS = [
    'wind_u',
    'wind_v',
    'temp',
    'press',
    'density',
    'precipitation',
]


In [ ]:
def add_wind_components(frame):
    """Convert meteorological wind direction into Cartesian vector components."""
    result = frame.copy()
    direction_rad = np.deg2rad(result['w_dir'])

    # Meteorological direction indicates where the wind comes from
    result['wind_u'] = -result['w_speed'] * np.sin(direction_rad)
    result['wind_v'] = -result['w_speed'] * np.cos(direction_rad)
    return result


def restore_wind_state(frame):
    """Restore wind speed and meteorological direction after component interpolation."""
    result = frame.copy()
    reconstructed_speed = np.hypot(result['wind_u'], result['wind_v'])
    reconstructed_direction = (
        np.rad2deg(np.arctan2(-result['wind_u'], -result['wind_v'])) + 360
    ) % 360

    # Existing hourly values are retained exactly; only missing quarter-hours are filled
    result['w_speed'] = result['w_speed'].fillna(reconstructed_speed).clip(lower=0)
    result['w_dir'] = result['w_dir'].fillna(reconstructed_direction) % 360
    result['precipitation'] = result['precipitation'].clip(lower=0)
    return result


weather_with_vectors = add_wind_components(clean_offshore_df)


### Empirical comparison on hidden hourly observations

The following diagnostic hides every fourth eligible observation within the sampled forecast runs. It reports:

- MAE for wind speed;
- circular MAE for wind direction;
- MAE for the point-wise power proxy.

The test is a relative comparison of interpolation rules, not proof of the true 15-minute trajectory. Weather dynamics between hours can still contain variation unavailable in the hourly ECMWF data.

In [ ]:
def circular_absolute_error_degrees(actual, predicted):
    """Return the shortest absolute angular distance in degrees."""
    return np.abs((predicted - actual + 180) % 360 - 180)


def validate_interpolation_methods(frame, validation_runs=40):
    """Compare meteorology-first and direct-proxy interpolation on hidden hourly rows."""
    run_ids = frame['dt_run_utc'].drop_duplicates().iloc[:validation_runs]
    sample = frame[frame['dt_run_utc'].isin(run_ids)].copy()
    records = []

    for (_, _), group in sample.groupby(['dt_run_utc', 'point_id'], sort=False):
        group = group.sort_values('dt_utc').set_index('dt_utc')
        if len(group) < 7:
            continue

        # Candidates must have exact hourly observations on both sides
        candidate_positions = np.arange(1, len(group) - 1)
        has_hourly_neighbours = (
            (group.index[candidate_positions] - group.index[candidate_positions - 1]
             == pd.Timedelta(hours=1))
            & (group.index[candidate_positions + 1] - group.index[candidate_positions]
               == pd.Timedelta(hours=1))
        )
        # Taking every fourth candidate prevents adjacent observations from being hidden
        hidden_positions = candidate_positions[has_hourly_neighbours][::4]
        if len(hidden_positions) == 0:
            continue
        hidden_index = group.index[hidden_positions]
        truth = group.loc[hidden_index]

        for method in ['time', 'pchip']:
            masked = group[CONTINUOUS_WEATHER_COLS].copy()
            masked.loc[hidden_index, CONTINUOUS_WEATHER_COLS] = np.nan
            reconstructed = masked.interpolate(method=method, limit_area='inside')

            predicted_speed = np.hypot(
                reconstructed.loc[hidden_index, 'wind_u'],
                reconstructed.loc[hidden_index, 'wind_v'],
            )
            predicted_direction = (
                np.rad2deg(
                    np.arctan2(
                        -reconstructed.loc[hidden_index, 'wind_u'],
                        -reconstructed.loc[hidden_index, 'wind_v'],
                    )
                ) + 360
            ) % 360
            predicted_proxy = (
                reconstructed.loc[hidden_index, 'density'] * predicted_speed ** 3
            )
            actual_proxy = truth['density'] * truth['w_speed'] ** 3

            for idx in hidden_index:
                records.append({
                    'strategy': f'meteorology_first_{method}',
                    'w_speed_abs_error': abs(predicted_speed.loc[idx] - truth.loc[idx, 'w_speed']),
                    'w_dir_abs_error': circular_absolute_error_degrees(
                        truth.loc[idx, 'w_dir'], predicted_direction.loc[idx]
                    ),
                    'power_proxy_abs_error': abs(predicted_proxy.loc[idx] - actual_proxy.loc[idx]),
                })

            # Direct interpolation of the engineered proxy is retained only as a benchmark
            proxy = group['density'] * group['w_speed'] ** 3
            proxy.loc[hidden_index] = np.nan
            direct_proxy = proxy.interpolate(method=method, limit_area='inside')

            for idx in hidden_index:
                records.append({
                    'strategy': f'direct_proxy_{method}',
                    'w_speed_abs_error': np.nan,
                    'w_dir_abs_error': np.nan,
                    'power_proxy_abs_error': abs(direct_proxy.loc[idx] - actual_proxy.loc[idx]),
                })

    validation_details = pd.DataFrame(records)
    validation_summary = (
        validation_details
        .groupby('strategy', as_index=False)
        .agg(
            hidden_observations=('power_proxy_abs_error', 'size'),
            w_speed_mae=('w_speed_abs_error', 'mean'),
            w_dir_circular_mae_deg=('w_dir_abs_error', 'mean'),
            power_proxy_mae=('power_proxy_abs_error', 'mean'),
        )
        .sort_values('power_proxy_mae')
        .reset_index(drop=True)
    )
    return validation_summary


interpolation_validation = validate_interpolation_methods(
    weather_with_vectors,
    validation_runs=min(VALIDATION_RUNS, weather_with_vectors['dt_run_utc'].nunique()),
)
interpolation_validation


### Interpolating the original meteorological variables

A new 15-minute index is created separately for each forecast run and measurement point. Synthetic timestamps are filled only when they are bracketed by **two consecutive original hourly observations exactly one hour apart**. This prevents interpolation across missing hours or gaps between forecast runs.

The `is_interpolated` flag is created from timestamp provenance, not inferred from values. Binary turbine-risk features are deliberately recalculated after interpolation rather than interpolated as fractions.

In [ ]:
def interpolate_run_point(group, method='time', frequency='15min'):
    """Interpolate one measurement point inside one forecast run without crossing gaps."""
    group = group.sort_values('dt_utc').drop_duplicates('dt_utc', keep='last')
    group = group.set_index('dt_utc')

    full_index = pd.date_range(
        start=group.index.min(),
        end=group.index.max(),
        freq=frequency,
        tz=group.index.tz,
    )
    result = group.reindex(full_index)
    result.index.name = 'dt_utc'

    is_original = result.index.isin(group.index)
    source_timestamp = pd.Series(pd.NaT, index=full_index, dtype=full_index.dtype)
    source_timestamp.loc[is_original] = full_index[is_original]

    previous_source = source_timestamp.ffill()
    next_source = source_timestamp.bfill()

    # Only fill quarter-hours between genuinely consecutive hourly source rows
    valid_interval = is_original | (
        (next_source - previous_source) == pd.Timedelta(hours=1)
    )

    interpolated_values = result[CONTINUOUS_WEATHER_COLS].interpolate(
        method=method,
        limit_area='inside',
    )
    result[CONTINUOUS_WEATHER_COLS] = interpolated_values.where(valid_interval)

    # These columns are constant inside one point and forecast run
    result['dt_run_utc'] = group['dt_run_utc'].iloc[0]
    result['point_id'] = group['point_id'].iloc[0]
    result['region'] = group['region'].iloc[0]
    result['dt_berlin'] = result.index.tz_convert('Europe/Berlin')
    result['is_interpolated'] = (~is_original).astype('int8')

    result = restore_wind_state(result)
    return result.reset_index()


interpolated_parts = []

for (_, _), point_run in weather_with_vectors.groupby(
    ['dt_run_utc', 'point_id'],
    sort=False,
):
    interpolated_parts.append(
        interpolate_run_point(
            point_run,
            method=INTERPOLATION_METHOD,
            frequency=INTERPOLATION_FREQUENCY,
        )
    )

interpolated_weather = (
    pd.concat(interpolated_parts, ignore_index=True)
    .dropna(subset=CONTINUOUS_WEATHER_COLS + ['w_speed', 'w_dir'])
    .sort_values(['dt_run_utc', 'dt_utc', 'point_id'])
    .reset_index(drop=True)
)

print(f"Interpolated long weather table size: {interpolated_weather.shape}")
print(interpolated_weather['is_interpolated'].value_counts().sort_index())


In [ ]:
# A short provenance and continuity check for one point and one forecast run
example_key = (
    interpolated_weather[['dt_run_utc', 'point_id']]
    .drop_duplicates()
    .iloc[0]
)

example_interpolation = interpolated_weather[
    (interpolated_weather['dt_run_utc'] == example_key['dt_run_utc'])
    & (interpolated_weather['point_id'] == example_key['point_id'])
][['dt_run_utc', 'dt_utc', 'point_id', 'w_speed', 'w_dir', 'is_interpolated']]

display(example_interpolation.head(9))


### Feature engineering after interpolation

The physical and binary features are now calculated from the reconstructed meteorological state:

- `power_proxy` is recalculated for every point from interpolated wind speed and density;
- wind direction is encoded with sine and cosine for circular regional statistics;
- turbine operating-risk flags are recalculated from the interpolated values;
- 14 point-wise proxies are pivoted into the same standard constructor used in the original notebook.

At an original timestamp, `is_interpolated=0` only when all contributing point observations are original. If any contributing point is synthetic, the final row is marked as interpolated.

In [ ]:
# Setting thresholds for various risk conditions
CUTIN_THRESHOLD = 3
CUTOUT_THRESHOLD = 25
SATURATION_THRESHOLD = 12
ICING_TEMP_UP_THRESHOLD = 2
ICING_TEMP_DOWN_THRESHOLD = -10

df = interpolated_weather.copy()

# Recalculating physical proxy and circular wind-direction components
df['power_proxy'] = (df['w_speed'] ** 3) * df['density']
df['w_dir_rad'] = np.deg2rad(df['w_dir'])
df['w_dir_sin'] = np.sin(df['w_dir_rad'])
df['w_dir_cos'] = np.cos(df['w_dir_rad'])

# Recalculating binary conditions after interpolation
df['cut_in_risk'] = (df['w_speed'] < CUTIN_THRESHOLD).astype('int8')
df['cut_out_risk'] = (df['w_speed'] >= CUTOUT_THRESHOLD).astype('int8')

saturation_condition = (
    (df['w_speed'] >= SATURATION_THRESHOLD)
    & (df['w_speed'] < CUTOUT_THRESHOLD)
)
df['saturation_risk'] = saturation_condition.astype('int8')

icing_condition = (
    (df['precipitation'] > 0)
    & (df['temp'] <= ICING_TEMP_UP_THRESHOLD)
    & (df['temp'] >= ICING_TEMP_DOWN_THRESHOLD)
)
df['icing_risk'] = icing_condition.astype('int8')


In [ ]:
# ---------------------------------------------------------
# Pivoting raw power_proxy values by 14 measurement points
# ---------------------------------------------------------
POINT_COL = 'point_id'
time_cols = ['dt_run_utc', 'dt_utc', 'dt_berlin']

proxy_wide = df.pivot_table(
    index=time_cols,
    columns=POINT_COL,
    values='power_proxy',
    aggfunc='first',
)

proxy_wide.columns = [
    f"point_{str(col).replace(' ', '_')}_power_proxy"
    for col in proxy_wide.columns
]
proxy_wide = proxy_wide.reset_index()

# ---------------------------------------------------------
# Aggregation of other features by regions
# ---------------------------------------------------------
agg_funcs = {
    'w_dir_sin': ['mean'],
    'w_dir_cos': ['mean'],
    'temp': ['mean'],
    'press': ['std'],
    'precipitation': ['mean'],
    'cut_in_risk': ['mean'],
    'cut_out_risk': ['mean'],
    'saturation_risk': ['mean'],
    'icing_risk': ['mean'],
}

grouped = (
    df.groupby(time_cols + ['region'])
    .agg(agg_funcs)
)

grouped.columns = ['_'.join(col).strip() for col in grouped.columns.values]
grouped = grouped.reset_index()

# Circular variance measures disagreement of point-wise wind directions
grouped['w_dir_circ_var'] = 1 - np.sqrt(
    grouped['w_dir_sin_mean'] ** 2 + grouped['w_dir_cos_mean'] ** 2
)

region_pivot = grouped.pivot(index=time_cols, columns='region')
region_pivot.columns = [
    f"{col[1].replace(' ', '_')}_{col[0]}"
    for col in region_pivot.columns
]
region_pivot = region_pivot.reset_index()

# A final row is original only if all contributing point rows are original
provenance = (
    df.groupby(time_cols, as_index=False)['is_interpolated']
    .max()
)

# Combining point-wise proxies, regional features, and provenance
feature_table = (
    proxy_wide
    .merge(region_pivot, on=time_cols, how='inner')
    .merge(provenance, on=time_cols, how='inner')
    .sort_values(['dt_run_utc', 'dt_utc'])
    .reset_index(drop=True)
)

# Cleaning column names for regions
feature_table.columns = feature_table.columns.str.replace(
    r'^Baltic_Sea_', 'baltic_', regex=True
)
feature_table.columns = feature_table.columns.str.replace(
    r'^North_Sea_', 'north_', regex=True
)


In [ ]:
# Checking the standard constructor
proxy_cols = [
    column for column in feature_table.columns
    if 'power_proxy' in column and 'delta' not in column
]

if len(proxy_cols) != 14:
    raise ValueError(
        f"Expected 14 point-wise power proxies, but found {len(proxy_cols)}"
    )

if 'minute_step' in feature_table.columns:
    raise ValueError("minute_step must not be present in the interpolated dataset")

print(f"Feature table size: {feature_table.shape}")
print(f"Columns with proxy values ({len(proxy_cols)}):")
print(proxy_cols)
print('\nInterpolation flag values:')
print(feature_table['is_interpolated'].value_counts().sort_index())

display(
    feature_table[
        ['dt_run_utc', 'dt_utc', 'dt_berlin', 'is_interpolated'] + proxy_cols[:3]
    ].head(8)
)


### Two dataset variants on the same target timeline

Both variants use exactly the same set of eligible hourly anchors as the original notebook. An anchor is retained only when the same forecast run contains observations exactly one hour before and one hour after it. Each retained anchor contributes its four interpolated timestamps: `:00`, `:15`, `:30`, and `:45`.

This reproduces the original non-overlapping target timeline without using a `minute_step` column. Consequently, neighbouring forecast runs do not contribute duplicate target timestamps.

For the optional delta variant, T−1 and T+1 are calculated on the original hourly anchors and attached to all four quarter-hours belonging to that anchor. This preserves the original one-hour delta semantics while the current weather features themselves remain interpolated at 15-minute resolution.

In [ ]:
def get_eligible_hourly_features(frame):
    """Keep original hourly anchors with exact T-1h and T+1h neighbours."""
    hourly = frame[frame['is_interpolated'].eq(0)].copy()
    eligible_parts = []

    for run_id, group in hourly.groupby('dt_run_utc', sort=False):
        group = group.sort_values('dt_utc').copy()

        has_t_minus_1 = group['dt_utc'].shift(1).eq(
            group['dt_utc'] - pd.Timedelta(hours=1)
        )
        has_t_plus_1 = group['dt_utc'].shift(-1).eq(
            group['dt_utc'] + pd.Timedelta(hours=1)
        )
        eligible_parts.append(group[has_t_minus_1 & has_t_plus_1])

    return pd.concat(eligible_parts, ignore_index=True)


def select_original_target_timeline(frame, eligible_hourly):
    """Select four interpolated quarter-hours for every eligible hourly anchor."""
    eligible_keys = (
        eligible_hourly[['dt_run_utc', 'dt_utc']]
        .rename(columns={'dt_utc': 'anchor_hour'})
    )

    result = frame.copy()
    result['anchor_hour'] = result['dt_utc'].dt.floor('h')
    result = result.merge(
        eligible_keys,
        on=['dt_run_utc', 'anchor_hour'],
        how='inner',
        validate='many_to_one',
    )
    return (
        result
        .drop(columns='anchor_hour')
        .sort_values(['dt_run_utc', 'dt_utc'])
        .reset_index(drop=True)
    )


def build_hourly_delta_table(eligible_hourly, all_hourly):
    """Calculate original-style hourly deltas for eligible anchor timestamps."""
    model_weather_cols = [
        column for column in all_hourly.columns
        if column not in time_cols + ['is_interpolated']
    ]
    delta_parts = []

    for run_id, group in all_hourly.groupby('dt_run_utc', sort=False):
        group = group.sort_values('dt_utc').copy()

        diff_t_minus_1 = group[model_weather_cols].shift(1) - group[model_weather_cols]
        diff_t_minus_1.columns = [
            f"{column}_delta_before" for column in model_weather_cols
        ]

        diff_t_plus_1 = group[model_weather_cols].shift(-1) - group[model_weather_cols]
        diff_t_plus_1.columns = [
            f"{column}_delta_after" for column in model_weather_cols
        ]

        combined = pd.concat(
            [
                group[['dt_run_utc', 'dt_utc']].reset_index(drop=True),
                diff_t_minus_1.reset_index(drop=True),
                diff_t_plus_1.reset_index(drop=True),
            ],
            axis=1,
        )
        delta_parts.append(combined)

    delta_table = pd.concat(delta_parts, ignore_index=True)
    eligible_keys = eligible_hourly[['dt_run_utc', 'dt_utc']]

    return (
        delta_table
        .merge(
            eligible_keys,
            on=['dt_run_utc', 'dt_utc'],
            how='inner',
            validate='one_to_one',
        )
        .rename(columns={'dt_utc': 'anchor_hour'})
    )


# Hourly rows reconstructed from original observations only
all_hourly_features = feature_table[
    feature_table['is_interpolated'].eq(0)
].copy()
eligible_hourly_features = get_eligible_hourly_features(feature_table)

# Both variants start from the same target timestamps
dataset_interpolated = select_original_target_timeline(
    feature_table,
    eligible_hourly_features,
)

hourly_delta_table = build_hourly_delta_table(
    eligible_hourly_features,
    all_hourly_features,
)

dataset_interpolated_with_t_deltas = dataset_interpolated.copy()
dataset_interpolated_with_t_deltas['anchor_hour'] = (
    dataset_interpolated_with_t_deltas['dt_utc'].dt.floor('h')
)
dataset_interpolated_with_t_deltas = (
    dataset_interpolated_with_t_deltas
    .merge(
        hourly_delta_table,
        on=['dt_run_utc', 'anchor_hour'],
        how='inner',
        validate='many_to_one',
    )
    .drop(columns='anchor_hour')
    .sort_values(['dt_run_utc', 'dt_utc'])
    .reset_index(drop=True)
)

if len(dataset_interpolated) != len(dataset_interpolated_with_t_deltas):
    raise AssertionError("Both dataset variants must use the same target timeline")

print(f"Eligible original hourly anchors: {len(eligible_hourly_features):,}")
print(f"Without deltas: {dataset_interpolated.shape}")
print(f"With T-1/T+1 deltas: {dataset_interpolated_with_t_deltas.shape}")

### Lead time and cyclical calendar features

Both variants receive the same lead-time and cyclical time features. The quarter-hour position is already represented continuously by `hour_sin` and `hour_cos`; therefore, a separate categorical `minute_step` feature is unnecessary.

In [ ]:
def add_time_features(frame):
    """Add forecast lead time and local cyclical calendar features."""
    result = frame.copy()

    result['lead_time'] = (
        result['dt_utc'] - result['dt_run_utc']
    ).dt.total_seconds() / 3600.0

    time_of_day_hours = (
        result['dt_berlin'].dt.hour
        + result['dt_berlin'].dt.minute / 60.0
    )
    result['hour_sin'] = np.sin(2 * np.pi * time_of_day_hours / 24)
    result['hour_cos'] = np.cos(2 * np.pi * time_of_day_hours / 24)

    day_of_year = result['dt_berlin'].dt.dayofyear
    result['doy_sin'] = np.sin(2 * np.pi * day_of_year / 365.25)
    result['doy_cos'] = np.cos(2 * np.pi * day_of_year / 365.25)
    return result


dataset_interpolated = add_time_features(dataset_interpolated)
dataset_interpolated_with_t_deltas = add_time_features(
    dataset_interpolated_with_t_deltas
)


### Creating the target for training

As in the original notebook, the target is the **Capacity Factor**:

`gen_wind_off_act / (capacity × 1000)`

Installed capacity is aligned backward because it is a low-frequency monthly series. Actual generation is joined by strict 15-minute timestamp matching.

Only rows without capacity or target values are removed here. Weather completeness is checked explicitly instead of calling `dropna(how='any')`, which makes accidental feature loss easier to diagnose.

In [ ]:
def attach_target(frame, capacity_frame, generation_frame):
    """Attach installed capacity, actual generation, and capacity factor."""
    result = frame.copy()
    capacity = capacity_frame.copy()
    generation = generation_frame.copy()

    # Normalising all merge keys to timezone-naive UTC
    result['dt_utc'] = (
        pd.to_datetime(result['dt_utc'], utc=True)
        .dt.tz_localize(None)
        .astype('datetime64[ns]')
    )
    capacity['dt_utc'] = (
        pd.to_datetime(capacity['dt_utc'], utc=True)
        .dt.tz_localize(None)
        .astype('datetime64[ns]')
    )
    generation.index = (
        pd.to_datetime(generation.index, utc=True)
        .tz_localize(None)
        .astype('datetime64[ns]')
    )

    result = pd.merge_asof(
        result.sort_values('dt_utc'),
        capacity.sort_values('dt_utc'),
        on='dt_utc',
        direction='backward',
    )
    result = result.rename(columns={'offshore_capacity': 'capacity'})
    result = result.set_index('dt_utc')

    result = result.join(
        generation[['gen_wind_off_act']],
        how='left',
    )
    result['capacity_factor'] = (
        result['gen_wind_off_act'] / (result['capacity'] * 1000)
    )

    # The model target must be observed; feature NaNs are treated as an error below
    result = result.dropna(subset=['capacity', 'gen_wind_off_act', 'capacity_factor'])

    feature_cols = [
        column for column in result.columns
        if column not in ['gen_wind_off_act', 'capacity_factor']
    ]
    missing_feature_count = int(result[feature_cols].isna().sum().sum())
    if missing_feature_count:
        raise ValueError(
            f"Found {missing_feature_count} missing feature values after target merge"
        )

    return result.sort_index()


dataset_interpolated = attach_target(
    dataset_interpolated,
    wind_offshore_capacity,
    actual_generation_data,
)
dataset_interpolated_with_t_deltas = attach_target(
    dataset_interpolated_with_t_deltas,
    wind_offshore_capacity,
    actual_generation_data,
)


In [ ]:
# Final integrity checks for both variants
for dataset_name, dataset in {
    'without_deltas': dataset_interpolated,
    'with_t_deltas': dataset_interpolated_with_t_deltas,
}.items():
    current_proxy_cols = [
        column for column in dataset.columns
        if 'power_proxy' in column and 'delta' not in column
    ]

    assert len(current_proxy_cols) == 14
    assert 'minute_step' not in dataset.columns
    assert dataset['is_interpolated'].isin([0, 1]).all()
    composite_keys = dataset.reset_index()[['dt_utc', 'dt_run_utc']]
    assert not composite_keys.duplicated().any()
    assert not dataset.index.duplicated().any()

    print(
        f"{dataset_name}: rows={len(dataset):,}, columns={dataset.shape[1]}, "
        f"period={dataset.index.min()} — {dataset.index.max()}"
    )

print('\nStandard dataset interpolation flag:')
print(dataset_interpolated['is_interpolated'].value_counts().sort_index())


### Saving both alternative datasets

The save commands are commented out to match the workflow of the original notebook and avoid overwriting artifacts during exploratory runs.

The standard constructor is the version **without deltas**. The second file provides the optional T−1/T+1 hourly-delta feature set.

In [ ]:
# Extracting date ranges for output file names
dataset_min_date = dataset_interpolated.index.min().strftime('%Y-%m-%d')
dataset_max_date = dataset_interpolated.index.max().strftime('%Y-%m-%d')

# # Uncomment if necessary============================================================
# # Saving the standard interpolated dataset without T-1/T+1
# dataset_interpolated.to_parquet(
#     DATASETS_DIR
#     / f'offshore_dataset_15m_interpolated_14_points_proxy_{dataset_min_date}_to_{dataset_max_date}.parquet'
# )

# # Saving the optional interpolated dataset with one-hour T-1/T+1 deltas
# dataset_interpolated_with_t_deltas.to_parquet(
#     DATASETS_DIR
#     / f'offshore_dataset_15m_interpolated_14_points_proxy_with_t_deltas_{dataset_min_date}_to_{dataset_max_date}.parquet'
# )


### Output of numerical data distributions

In [ ]:
dataset_interpolated.describe()


In [ ]:
dataset_interpolated_with_t_deltas.describe()


### Final Dataset Description

At this stage, two clean alternative datasets are ready for model training.

The standard version contains 14 individual `power_proxy` signals, regional weather aggregates, continuous calendar features, lead time, and the binary `is_interpolated` provenance flag. It does not contain `minute_step` or T−1/T+1 deltas.

The optional version adds original-style T−1/T+1 differences calculated at the hourly anchor and attached to all four interpolated quarter-hours belonging to it. It is intended for a direct ablation test against the standard version on exactly the same target timeline.

The interpolation method is controlled by `INTERPOLATION_METHOD`. PCHIP is the current default because it performs better in the masked-hour diagnostic on this history. Linear time interpolation remains the conservative fallback, and the final choice should also be checked by downstream time-based model validation.